# Análisis Exploratorio de Datos (EDA)

**Responsable:** Edwardo Mejia  

El objetivo de este notebook es revisar la calidad del conjunto de datos, interpretar sus estadísticas y observar patrones o relaciones antes del entrenamiento de los modelos. En esta etapa todavía no se entrena ningún algoritmo.

## 1. Importación de librerías y carga de datos

Se utilizan `pandas` para trabajar con el dataset, `numpy` para operaciones numéricas y `matplotlib`/`seaborn` para las gráficas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/tsla_processed.csv')
df.head()

> **Nota:** el CSV procesado no conserva una columna de fecha. Por esa razón, la evolución temporal se representa mediante el orden cronológico de las filas. No se inventaron fechas.

## 2. Revisión general del dataset

In [ ]:
print('Filas y columnas:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isnull().sum())
print('\nRegistros duplicados:', df.duplicated().sum())

**Interpretación:** el dataset tiene **1,624 registros y 12 columnas**. No presenta valores nulos ni registros duplicados. Esto indica que la limpieza realizada en la etapa anterior dejó el archivo listo para el análisis. Las once características están estandarizadas, mientras que `precio_futuro` se mantiene en dólares.

## 3. Estadísticas descriptivas

In [ ]:
estadisticas = df.describe().T
estadisticas['mediana'] = df.median(numeric_only=True)
estadisticas

**Interpretación:** el precio futuro tiene una media de **US$493.29**, una mediana de **US$400.56** y una desviación estándar de **US$323.58**. El mínimo es **US$108.10** y el máximo **US$2238.75**. La distancia entre estos valores confirma que el precio atravesó etapas muy diferentes.

## 4. Evolución del precio futuro

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(x=np.arange(len(df)), y=df['precio_futuro'])
plt.title('Evolución del precio futuro de TSLA')
plt.xlabel('Orden cronológico de las observaciones')
plt.ylabel('Precio futuro (USD)')
plt.tight_layout()
plt.show()

**Por qué importa:** permite identificar tendencias, cambios de nivel y períodos de variación fuerte. La gráfica muestra que el precio no sigue un comportamiento constante; por tanto, los modelos tendrán que aprender de diferentes etapas del mercado.

## 5. Distribución y valores extremos del precio

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df['precio_futuro'], bins=35, kde=True, color='steelblue')
plt.title('Distribución del precio futuro')
plt.xlabel('Precio futuro (USD)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,3.8))
sns.boxplot(x=df['precio_futuro'], color='steelblue')
plt.title('Diagrama de caja del precio futuro')
plt.xlabel('Precio futuro (USD)')
plt.tight_layout()
plt.show()

**Interpretación:** el 50 % central de los precios está entre **US$240.98 y US$705.64**. El histograma no es simétrico y presenta una extensión hacia precios altos. Los puntos alejados del boxplot no se eliminan automáticamente porque pueden ser movimientos reales de TSLA.

## 6. Retorno diario y volumen

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df['retorno_diario'], bins=40, kde=True, color='steelblue')
plt.title('Distribución del retorno diario estandarizado')
plt.xlabel('Retorno diario estandarizado')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,3.8))
sns.boxplot(x=df['volume'],color='steelblue')
plt.title('Diagrama de caja del volumen estandarizado')
plt.xlabel('Volumen estandarizado')
plt.tight_layout()
plt.show()

**Interpretación:** los retornos se concentran alrededor del promedio, pero existen jornadas con cambios más extremos. El volumen también contiene valores alejados, que pueden representar días con una actividad bursátil fuera de lo habitual.

## 7. Matriz de correlación

In [ ]:
correlacion = df.corr(numeric_only=True)

plt.figure(figsize=(13,10))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 7})
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

correlacion_objetivo = (
    correlacion['precio_futuro']
    .drop('precio_futuro')
    .sort_values(ascending=False)
)
correlacion_objetivo

In [ ]:
plt.figure(figsize=(9,6))
correlacion_objetivo.sort_values().plot(kind='barh')
plt.title('Correlación con el precio futuro')
plt.xlabel('Coeficiente de correlación')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

**Interpretación:** las correlaciones más altas con `precio_futuro` corresponden a `close` (0.987), `low` (0.986), `high` (0.985) y `open` (0.984). También se destacan `precio_anterior`, `sma_5` y `sma_20`. El volumen y su promedio móvil presentan correlaciones negativas. Esto describe una asociación lineal dentro del período, pero no significa causalidad.

## 8. Gráficas de dispersión

In [ ]:
variables = ['close', 'precio_anterior', 'sma_5', 'volume']

for variable in variables:
    plt.figure(figsize=(7.5,5))
    sns.scatterplot(data=df, x=variable, y='precio_futuro', alpha=0.45, s=25)
    plt.title(f'{variable} frente a precio_futuro')
    plt.xlabel(f'{variable} (estandarizada)')
    plt.ylabel('Precio futuro (USD)')
    plt.tight_layout()
    plt.show()

**Interpretación:** `close`, `precio_anterior` y `sma_5` muestran patrones ascendentes bastante definidos. En `volume` se observa una relación inversa y más dispersa. La dispersión permite comprobar visualmente la forma de las relaciones que aparecen en la tabla de correlaciones.

## 9. Volatilidad frente a retorno

In [ ]:
plt.figure(figsize=(7.5,5))
sns.scatterplot(data=df, x='volatilidad_5', y='retorno_diario', alpha=0.45, s=25)
plt.title('Volatilidad de 5 días frente a retorno diario')
plt.xlabel('Volatilidad estandarizada')
plt.ylabel('Retorno diario estandarizado')
plt.tight_layout()
plt.show()

**Interpretación:** esta gráfica ayuda a revisar si una mayor inestabilidad reciente coincide con movimientos diarios más extremos. La volatilidad aporta información sobre el riesgo, aunque no determina por sí sola la dirección del precio.

## 10. Conclusión del EDA

El dataset está limpio y cumple con el tamaño solicitado. Las variables más relacionadas con el precio futuro son las vinculadas al nivel reciente del precio: apertura, máximo, mínimo, cierre, precio anterior y medias móviles. El retorno, la volatilidad y el volumen aportan una perspectiva diferente sobre los cambios, el riesgo y la actividad del mercado.

Los valores extremos se conservaron porque pueden representar movimientos reales. Con estos resultados, el conjunto de datos puede continuar a la etapa de entrenamiento y comparación de modelos.